In [17]:
# Final retry: use outer triple-double-quotes and ensure all inner multilines use triple-single-quotes.
import nbformat, os, textwrap
from nbformat.v4 import new_code_cell

path = r"C:\Users\lakshmisha\Desktop\crop-weed-detection"
nb = nbformat.read(path, as_version=4)

flask_code = textwrap.dedent("""
# --- Simple Flask user system: login, register, home, result, profile, admin, sign out ---
# Save this cell and run it in a Jupyter environment (or copy it to a .py file) to start the app.
# It creates a local SQLite DB 'users.db' in the notebook directory.

from flask import Flask, request, session, redirect, url_for, render_template_string, g, abort
import sqlite3
from werkzeug.security import generate_password_hash, check_password_hash
import os

DB_PATH = os.path.join(os.getcwd(), "users.db")

def get_db():
    db = getattr(g, "_database", None)
    if db is None:
        db = g._database = sqlite3.connect(DB_PATH)
        db.row_factory = sqlite3.Row
    return db

def init_db():
    db = sqlite3.connect(DB_PATH)
    cur = db.cursor()
    cur.execute(\"'''\")
    cur.execute('''CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password_hash TEXT NOT NULL,
            fullname TEXT,
            is_admin INTEGER DEFAULT 0
        )''')
    cur.execute('''CREATE TABLE IF NOT EXISTS results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            title TEXT NOT NULL,
            score REAL,
            FOREIGN KEY(user_id) REFERENCES users(id)
        )''')
    db.commit()
    db.close()

app = Flask(__name__)
app.secret_key = os.environ.get("FLASK_SECRET", "dev-secret-key-change-me")

NAV = \"\"\"<div style='background:#f2f2f2;padding:10px;margin-bottom:15px;'>
  <a href='/'>Home</a> | <a href='/result'>Results</a> | <a href='/profile'>Profile</a> | 
  <a href='/register'>Register</a> | <a href='/login'>Login</a> | <a href='/admin'>Admin</a> | <a href='/signout'>Sign out</a>
</div>\"\"\"

def render(page_title, body_html):
    return render_template_string('''
    <!doctype html><html><head><title>{{title}}</title></head>
    <body>
      {{nav|safe}}
      <h2>{{title}}</h2>
      <div>{{body|safe}}</div>
    </body></html>
    ''', title=page_title, body=body_html, nav=NAV)

@app.teardown_appcontext
def close_conn(exception):
    db = getattr(g, "_database", None)
    if db is not None:
        db.close()

@app.route("/")
def home():
    user = session.get("user")
    welcome = f"<p>Welcome, {user['fullname'] if user else 'guest'}!</p>"
    if user:
        welcome += f"<p>Logged in as <strong>{user['username']}</strong>. <a href='/profile'>Go to profile</a>.</p>"
    else:
        welcome += "<p>Please <a href='/login'>login</a> or <a href='/register'>register</a>.</p>"
    return render("Home", welcome)

@app.route("/register", methods=["GET", "POST"])
def register():
    if request.method == "POST":
        username = request.form.get("username","").strip()
        fullname = request.form.get("fullname","").strip()
        password = request.form.get("password","")
        if not username or not password:
            return render("Register", "<p style='color:red;'>Username and password required.</p>" + register_form())
        db = get_db()
        cur = db.cursor()
        try:
            cur.execute("INSERT INTO users (username, password_hash, fullname) VALUES (?, ?, ?)", 
                        (username, generate_password_hash(password), fullname))
            db.commit()
            return redirect(url_for("login"))
        except sqlite3.IntegrityError:
            return render("Register", "<p style='color:red;'>Username already taken.</p>" + register_form())
    return render("Register", register_form())

def register_form():
    return '''<form method="post">
      <label>Username: <input name="username"></label><br>
      <label>Full name: <input name="fullname"></label><br>
      <label>Password: <input type="password" name="password"></label><br>
      <button type="submit">Register</button>
    </form>'''

@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        username = request.form.get("username","").strip()
        password = request.form.get("password","")
        db = get_db()
        cur = db.cursor()
        cur.execute("SELECT * FROM users WHERE username = ?", (username,))
        row = cur.fetchone()
        if row and check_password_hash(row["password_hash"], password):
            session["user"] = {"id": row["id"], "username": row["username"], "fullname": row["fullname"], "is_admin": bool(row["is_admin"])}
            return redirect(url_for("home"))
        else:
            return render("Login", "<p style='color:red;'>Invalid credentials.</p>" + login_form())
    return render("Login", login_form())

def login_form():
    return '''<form method="post">
      <label>Username: <input name="username"></label><br>
      <label>Password: <input type="password" name="password"></label><br>
      <button type="submit">Login</button>
    </form>'''

@app.route("/profile")
def profile():
    user = session.get("user")
    if not user:
        return redirect(url_for("login"))
    db = get_db()
    cur = db.cursor()
    cur.execute("SELECT * FROM results WHERE user_id = ?", (user["id"],))
    results = cur.fetchall()
    body = f"<p>Username: {user['username']}<br>Full name: {user['fullname'] or ''}</p>"
    if results:
        body += "<h4>Your results</h4><ul>"
        for r in results:
            body += f"<li>{r['title']} — score: {r['score']}</li>"
        body += "</ul>"
    else:
        body += "<p>No results yet.</p>"
    return render("Profile", body)

@app.route("/result")
def result():
    db = get_db()
    cur = db.cursor()
    cur.execute("SELECT r.id, r.title, r.score, u.username FROM results r JOIN users u ON r.user_id = u.id ORDER BY r.id DESC LIMIT 20")
    rows = cur.fetchall()
    body = "<h3>Recent results</h3>"
    if rows:
        body += "<ul>"
        for r in rows:
            body += f"<li>{r['title']} — {r['score']} (user: {r['username']})</li>"
        body += "</ul>"
    else:
        body += "<p>No results available.</p>"
    return render("Results", body)

@app.route("/admin", methods=["GET","POST"])
def admin():
    user = session.get("user")
    if not user:
        return redirect(url_for("login"))
    if not user.get("is_admin"):
        abort(403, description="Admin access required")
    db = get_db()
    cur = db.cursor()
    if request.method == "POST":
        username = request.form.get("username","").strip()
        title = request.form.get("title","").strip()
        score = request.form.get("score","").strip()
        cur.execute("SELECT id FROM users WHERE username = ?", (username,))
        row = cur.fetchone()
        if not row:
            return render("Admin", "<p style='color:red;'>User not found.</p>" + admin_form())
        try:
            cur.execute("INSERT INTO results (user_id, title, score) VALUES (?, ?, ?)", (row["id"], title, float(score)))
            db.commit()
            return redirect(url_for("admin"))
        except Exception as e:
            return render("Admin", f"<p style='color:red;'>Error: {e}</p>" + admin_form())
    cur.execute("SELECT id, username, fullname, is_admin FROM users ORDER BY id ASC")
    users = cur.fetchall()
    cur.execute("SELECT r.id, r.title, r.score, u.username FROM results r JOIN users u ON r.user_id = u.id ORDER BY r.id DESC LIMIT 50")
    rows = cur.fetchall()
    body = "<h4>Users</h4><ul>"
    for u in users:
        body += f"<li>{u['username']} - {u['fullname'] or ''} - {'admin' if u['is_admin'] else 'user'}</li>"
    body += "</ul>"
    body += admin_form()
    body += "<h4>Recent results</h4><ul>"
    for r in rows:
        body += f"<li>{r['title']} — {r['score']} (user: {r['username']})</li>"
    body += "</ul>"
    return render("Admin", body)

def admin_form():
    return '''<form method="post">
      <label>Username (to assign result to): <input name="username"></label><br>
      <label>Title: <input name="title"></label><br>
      <label>Score: <input name="score"></label><br>
      <button type="submit">Create result</button>
    </form>'''

@app.route("/signout")
def signout():
    session.pop("user", None)
    return redirect(url_for("home"))

# Initialize DB and ensure admin user exists with password 'adminpass' (change for production)
if not os.path.exists(DB_PATH):
    init_db()
# create admin if missing
db = sqlite3.connect(DB_PATH)
cur = db.cursor()
cur.execute("SELECT id FROM users WHERE username = ?", ("admin",))
if not cur.fetchone():
    cur.execute("INSERT INTO users (username, password_hash, fullname, is_admin) VALUES (?, ?, ?, ?)", 
                ("admin", generate_password_hash("adminpass"), "Administrator", 1))
    db.commit()
db.close()

print("Flask app cell added. Default admin: admin / adminpass. Database file: users.db") 
""")

nb.cells.append(new_code_cell(flask_code))
nbformat.write(nb, path)

path



FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\lakshmisha\\Desktop\\crop-weed-detection'